# 01 — Intégration et nettoyage des données

Ce notebook prépare les quatre sources du projet :

- D1 — inscriptions / profil ;
- D2 — T0 ;
- D3 — T1 ;
- D4 — T2.

Objectifs :

1. charger les quatre sources ;
2. préserver les données brutes ;
3. normaliser les identifiants ;
4. isoler les observations non appariables ;
5. conserver le parcours d'origine de T1 et T2 ;
6. construire une base longitudinale à une ligne par apprenant ;
7. matérialiser les cohortes M_T0 et M_T1 ;
8. produire les fichiers nécessaires aux notebooks suivants.

Aucune imputation, aucun scoring pédagogique et aucun modèle ML ne sont réalisés ici.

In [1]:
# Cellule 1 — Imports

In [2]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

print("✅ Environnement prêt")
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)

✅ Environnement prêt
Pandas : 2.1.4
NumPy  : 1.26.4


# Cellule 2 — Dossiers du projet

In [3]:
# ============================================================
# CELLULE 2 — Organisation du projet
# ============================================================

def trouver_racine_projet():

    candidats = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]

    for racine in candidats:

        if (racine / "data").exists():
            return racine.resolve()

    raise FileNotFoundError(
        "Impossible de trouver le dossier data/"
    )


PROJECT_ROOT = trouver_racine_projet()

DATA_DIR = PROJECT_ROOT / "data"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"

# Sous-dossier spécifique au notebook 01
OUTPUT_DIR = OUTPUT_ROOT / "01_nettoyage"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)

PROJECT_ROOT : /home/andrycomputer/Documents/Model-eval-pers-elearning
DATA_DIR     : /home/andrycomputer/Documents/Model-eval-pers-elearning/data
OUTPUT_DIR   : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/01_nettoyage


# Cellule 3 — Détection des quatre fichiers

In [4]:
# ============================================================
# CELLULE 3 — Détection automatique des sources
# ============================================================

def trouver_fichier(
    noms_exacts,
    motifs=None,
    extension=None
):

    # Noms exacts d'abord
    for nom in noms_exacts:

        chemin = DATA_DIR / nom

        if chemin.exists():
            return chemin


    motifs = [
        str(m).lower()
        for m in (motifs or [])
    ]


    # Recherche secondaire
    for chemin in sorted(DATA_DIR.iterdir()):

        if not chemin.is_file():
            continue

        if (
            extension is not None
            and chemin.suffix.lower() != extension.lower()
        ):
            continue

        nom = chemin.name.lower()

        if motifs and all(m in nom for m in motifs):
            return chemin


    raise FileNotFoundError(
        f"Fichier introuvable dans {DATA_DIR}"
    )


D1_FILE = trouver_fichier(
    [
        "inscriptions(3).json",
        "inscriptions.json"
    ],
    motifs=["inscription"],
    extension=".json"
)


D2_FILE = trouver_fichier(
    [
        "tests(3).json",
        "tests.json",
        "test_initial.json"
    ],
    motifs=["test"],
    extension=".json"
)


D3_FILE = trouver_fichier(
    [
        "mi-test.xlsx",
        "mi_test.xlsx"
    ],
    motifs=["mi", "test"],
    extension=".xlsx"
)


D4_FILE = trouver_fichier(
    [
        "RESULTATS_FINAL_TESTS_PAR_PARCOURS(1).xlsx",
        "RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx",
        "tests_finaux.xlsx"
    ],
    motifs=["final"],
    extension=".xlsx"
)


print("✅ Sources détectées")

print("D1 :", D1_FILE.name)
print("D2 :", D2_FILE.name)
print("D3 :", D3_FILE.name)
print("D4 :", D4_FILE.name)

✅ Sources détectées
D1 : inscriptions.json
D2 : tests.json
D3 : mi-test.xlsx
D4 : RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx


#  Cellule 4 — Fonctions utilitaires

In [5]:
# ============================================================
# CELLULE 4 — Fonctions utilitaires
# ============================================================

def charger_json_table(chemin):

    with open(
        chemin,
        "r",
        encoding="utf-8-sig"
    ) as f:

        obj = json.load(f)


    if isinstance(obj, list):

        return pd.json_normalize(obj)


    if isinstance(obj, dict):

        # dictionnaire de dictionnaires
        if (
            obj
            and all(
                isinstance(v, dict)
                for v in obj.values()
            )
        ):

            return pd.json_normalize(
                list(obj.values())
            )


        # dictionnaire contenant une liste
        for cle, valeur in obj.items():

            if (
                isinstance(valeur, list)
                and len(valeur) > 0
                and isinstance(valeur[0], dict)
            ):

                print(
                    "ℹ️ Données JSON dans la clé :",
                    cle
                )

                return pd.json_normalize(valeur)


        return pd.json_normalize(obj)


    raise ValueError(
        f"Structure JSON inconnue : {chemin}"
    )


def normaliser_nom_colonne(texte):

    texte = unicodedata.normalize(
        "NFKD",
        str(texte)
    )

    texte = "".join(
        c for c in texte
        if not unicodedata.combining(c)
    )

    texte = texte.lower().strip()

    return re.sub(
        r"[^a-z0-9]",
        "",
        texte
    )


def trouver_colonne_identifiant(df):

    candidats = [
        "identification",
        "identifiant",
        "studentid",
        "apprenantid",
        "idapprenant",
        "userid"
    ]

    mapping = {
        normaliser_nom_colonne(col): col
        for col in df.columns
    }


    for candidat in candidats:

        if candidat in mapping:
            return mapping[candidat]


    for nom_normalise, nom_original in mapping.items():

        if "identif" in nom_normalise:
            return nom_original


    return None


def nettoyer_identifiant(serie):

    s = (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
    )

    return s.replace(
        [
            "",
            "NAN",
            "NONE",
            "NULL",
            "<NA>"
        ],
        pd.NA
    )


def extraire_numero_id(valeur):

    if pd.isna(valeur):
        return np.nan

    match = re.search(
        r"(\d+)$",
        str(valeur)
    )

    if match:
        return int(match.group(1))

    return np.nan


def normaliser_valeur_objet(valeur):

    if isinstance(valeur, list):

        valeurs = [
            normaliser_valeur_objet(x)
            for x in valeur
        ]

        valeurs = [
            str(x)
            for x in valeurs
            if not pd.isna(x)
        ]

        return (
            " | ".join(valeurs)
            if valeurs
            else pd.NA
        )


    if isinstance(valeur, dict):

        return json.dumps(
            valeur,
            ensure_ascii=False,
            sort_keys=True
        )


    if isinstance(valeur, str):

        valeur = re.sub(
            r"\s+",
            " ",
            valeur
        ).strip()

        if valeur.upper() in {
            "",
            "NAN",
            "NONE",
            "NULL",
            "<NA>"
        }:

            return pd.NA

        return valeur


    return valeur


def normaliser_colonnes_objet(df):

    resultat = df.copy()

    for col in resultat.select_dtypes(
        include=["object"]
    ).columns:

        resultat[col] = (
            resultat[col]
            .map(normaliser_valeur_objet)
        )

    return resultat


print("✅ Fonctions prêtes")

✅ Fonctions prêtes


#  Cellule 5 — Chargement des données

In [6]:
# ============================================================
# CELLULE 5 — Chargement
# ============================================================

d1_raw = charger_json_table(D1_FILE)

d2_raw = charger_json_table(D2_FILE)


d3_sheets_raw = pd.read_excel(
    D3_FILE,
    sheet_name=None
)


d4_sheets_raw = pd.read_excel(
    D4_FILE,
    sheet_name=None
)


print("D1 :", d1_raw.shape)

print("D2 :", d2_raw.shape)


print("\nD3")

for nom, df in d3_sheets_raw.items():

    print(
        f"{nom:20s}",
        df.shape
    )


print("\nD4")

for nom, df in d4_sheets_raw.items():

    print(
        f"{nom:20s}",
        df.shape
    )

D1 : (472, 19)
D2 : (315, 25)

D3
MI-TEST-DA           (37, 24)
MI-TEST-BI           (65, 24)
MI-TEST-DS           (46, 24)
MI-TEST-IA           (62, 24)

D4
FINAL-TEST-DA        (34, 22)
FINAL-TEST-BI        (60, 22)
FINAL-TEST-DS        (48, 22)
FINAL-TEST-IA        (64, 22)


# Cellule 6 — Vérifier les feuilles

In [7]:
# ============================================================
# CELLULE 6 — Validation des feuilles
# ============================================================

T1_ATTENDUES = {
    "MI-TEST-DA",
    "MI-TEST-BI",
    "MI-TEST-DS",
    "MI-TEST-IA"
}


T2_ATTENDUES = {
    "FINAL-TEST-DA",
    "FINAL-TEST-BI",
    "FINAL-TEST-DS",
    "FINAL-TEST-IA"
}


manquantes_t1 = (
    T1_ATTENDUES
    - set(d3_sheets_raw)
)


manquantes_t2 = (
    T2_ATTENDUES
    - set(d4_sheets_raw)
)


if manquantes_t1:

    warnings.warn(
        f"Feuilles T1 manquantes : "
        f"{manquantes_t1}"
    )


if manquantes_t2:

    warnings.warn(
        f"Feuilles T2 manquantes : "
        f"{manquantes_t2}"
    )


print("✅ Structure Excel vérifiée")

✅ Structure Excel vérifiée


#  Cellule 7 — Concaténer T1 et T2

In [8]:
# ============================================================
# CELLULE 7 — Concaténation T1 / T2
# ============================================================

def detecter_parcours(nom_feuille):

    nom = str(
        nom_feuille
    ).upper()


    for parcours in [
        "DA",
        "BI",
        "DS",
        "IA"
    ]:

        if re.search(
            rf"(^|[-_\s]){parcours}($|[-_\s])",
            nom
        ):

            return parcours


    return None


def concatener_feuilles(
    feuilles,
    etape
):

    tables = []


    for nom_feuille, df in feuilles.items():

        temp = df.copy()

        temp["SOURCE_SHEET"] = nom_feuille

        temp["ETAPE"] = etape

        temp["PARCOURS"] = (
            detecter_parcours(
                nom_feuille
            )
        )

        tables.append(temp)


    return pd.concat(
        tables,
        ignore_index=True,
        sort=False
    )


t1_raw = concatener_feuilles(
    d3_sheets_raw,
    "T1"
)


t2_raw = concatener_feuilles(
    d4_sheets_raw,
    "T2"
)


print(
    "T1 concaténé :",
    t1_raw.shape
)

print(
    "T2 concaténé :",
    t2_raw.shape
)

T1 concaténé : (210, 72)
T2 concaténé : (206, 70)


#  Cellule 8 — Normalisation des IDs

In [9]:
# ============================================================
# CELLULE 8 — Identifiants
# ============================================================

ID_D1 = trouver_colonne_identifiant(
    d1_raw
)

ID_D2 = trouver_colonne_identifiant(
    d2_raw
)

ID_T1 = trouver_colonne_identifiant(
    t1_raw
)

ID_T2 = trouver_colonne_identifiant(
    t2_raw
)


print("D1 :", ID_D1)
print("D2 :", ID_D2)
print("T1 :", ID_T1)
print("T2 :", ID_T2)


if any(
    x is None
    for x in [
        ID_D1,
        ID_D2,
        ID_T1,
        ID_T2
    ]
):

    raise ValueError(
        "Identifiant manquant dans une source."
    )


d1 = normaliser_colonnes_objet(
    d1_raw
)

d2 = normaliser_colonnes_objet(
    d2_raw
)

t1 = normaliser_colonnes_objet(
    t1_raw
)

t2 = normaliser_colonnes_objet(
    t2_raw
)


d1["_ID_NORM"] = nettoyer_identifiant(
    d1[ID_D1]
)

d2["_ID_NORM"] = nettoyer_identifiant(
    d2[ID_D2]
)

t1["_ID_NORM"] = nettoyer_identifiant(
    t1[ID_T1]
)

t2["_ID_NORM"] = nettoyer_identifiant(
    t2[ID_T2]
)


for df in [
    d1,
    d2,
    t1,
    t2
]:

    df["_ID_NUM"] = (
        df["_ID_NORM"]
        .map(extraire_numero_id)
    )


print("✅ IDs normalisés")

D1 : IDENTIFICATION
D2 : Identification
T1 : IDENTIFICATION
T2 : IDENTIFICATION
✅ IDs normalisés


#  Cellule 9 — Nettoyage D1

In [10]:
# ============================================================
# CELLULE 9 — D1
# ============================================================

d1_clean = d1[
    d1["_ID_NORM"].notna()
].copy()


COL_ANNEE = None


for col in d1_clean.columns:

    if (
        normaliser_nom_colonne(col)
        == "anneedenaissance"
    ):

        COL_ANNEE = col
        break


if COL_ANNEE is not None:

    annee = pd.to_numeric(
        d1_clean[COL_ANNEE],
        errors="coerce"
    )


    ANNEE_REFERENCE = 2026
    annee_actuelle = ANNEE_REFERENCE


    d1_clean[
        "ANNEE_NAISSANCE_NUM"
    ] = annee


    d1_clean[
        "ANNEE_NAISSANCE_INVALIDE"
    ] = (
        annee.notna()
        & (
            (annee < 1900)
            | (annee > annee_actuelle)
        )
    )


    d1_clean.loc[
        d1_clean[
            "ANNEE_NAISSANCE_INVALIDE"
        ],
        "ANNEE_NAISSANCE_NUM"
    ] = np.nan


    print(
        "Années invalides :",
        d1_clean[
            "ANNEE_NAISSANCE_INVALIDE"
        ].sum()
    )


print(
    "D1 nettoyé :",
    d1_clean.shape
)


# Suppression de l'identité directe
d1_clean = d1_clean.drop(
    columns=[
        col
        for col in [
            "Title"
        ]
        if col in d1_clean.columns
    ]
)

print(
    "✅ Identité nominative supprimée des données analytiques"
)

Années invalides : 9
D1 nettoyé : (472, 23)
✅ Identité nominative supprimée des données analytiques


#  Cellule 10 — Questions T0

In [11]:
# ============================================================
# CELLULE 10 — Questions T0
# ============================================================

def numero_question_t0(col):

    match = re.match(
        r"^Q(\d+)\s*-",
        str(col).strip(),
        flags=re.IGNORECASE
    )

    if match:
        return int(
            match.group(1)
        )

    return None


question_cols_t0 = [
    col
    for col in d2.columns
    if numero_question_t0(col)
    is not None
]


question_cols_t0 = sorted(
    question_cols_t0,
    key=numero_question_t0
)


print(
    "Questions T0 détectées :",
    len(question_cols_t0)
)


def domaine_question(q):

    if 1 <= q <= 5:
        return "DA"

    if 6 <= q <= 10:
        return "BI"

    if 11 <= q <= 15:
        return "DS"

    if 16 <= q <= 20:
        return "IA"

    return None


dictionnaire_t0 = pd.DataFrame([
    {
        "Q_NUM":
            numero_question_t0(col),

        "DOMAINE":
            domaine_question(
                numero_question_t0(col)
            ),

        "QUESTION":
            col
    }

    for col in question_cols_t0
])


display(dictionnaire_t0)

Questions T0 détectées : 20


,Q_NUM,DOMAINE,QUESTION
0,1,DA,Q1 - Resume ventes Excel
1,2,DA,Q2 - Import CSV Excel
2,3,DA,Q3 - Modele donnees Excel
3,4,DA,Q4 - Mauvaise pratique visu Excel
4,5,DA,Q5 - 2e grande valeur Excel
5,6,BI,Q6 - Mesure vs Colonne DAX
6,7,BI,Q7 - CA annee precedente DAX
7,8,BI,Q8 - Vue Modele Power BI
8,9,BI,Q9 - Acces directeurs regionaux
9,10,BI,Q10 - 12 commerciaux 3 indicateurs


# Cellule 11 — Complétude et nettoyage T0

In [12]:
# ============================================================
# CELLULE 11 — T0
# ============================================================

d2["T0_NB_REPONSES"] = (
    d2[question_cols_t0]
    .notna()
    .sum(axis=1)
)


d2["T0_TAUX_COMPLETUDE"] = (
    d2["T0_NB_REPONSES"]
    / len(question_cols_t0)
    * 100
)


t0_clean = d2[
    d2["_ID_NORM"].notna()
].copy()


t0_sans_id = d2[
    d2["_ID_NORM"].isna()
].copy()


print(
    "T0 analytique :",
    len(t0_clean)
)

print(
    "T0 sans ID :",
    len(t0_sans_id)
)


display(
    t0_sans_id[
        [
            c
            for c in [
                "CreatedAt",
                "Date du test",
                "T0_NB_REPONSES",
                "T0_TAUX_COMPLETUDE"
            ]
            if c in t0_sans_id.columns
        ]
    ]
)

T0 analytique : 305
T0 sans ID : 10


,CreatedAt,Date du test,T0_NB_REPONSES,T0_TAUX_COMPLETUDE
0,2026-03-31 17:09:03+00:00,2026-03-31 17:09:02+00:00,20,100.0
1,2026-03-31 17:41:49+00:00,2026-03-31 17:41:48+00:00,20,100.0
2,2026-03-31 18:19:08+00:00,2026-03-31 18:19:07+00:00,20,100.0
3,2026-03-31 18:25:08+00:00,2026-03-31 18:25:06+00:00,20,100.0
4,2026-03-31 20:12:55+00:00,2026-03-31 20:12:54+00:00,20,100.0
5,2026-03-31 20:43:51+00:00,2026-03-31 20:43:50+00:00,20,100.0
6,2026-03-31 20:51:13+00:00,2026-03-31 20:51:13+00:00,20,100.0
7,2026-03-31 22:16:12+00:00,2026-03-31 22:16:11+00:00,20,100.0
18,2026-04-01 08:18:18+00:00,2026-04-01 08:18:15+00:00,20,100.0
222,2026-04-21 13:16:05+00:00,None,0,0.0


# Cellule 12 — Détecter les questions T1/T2

In [13]:
# ============================================================
# CELLULE 12 — Questions T1 / T2
# ============================================================

META_COLS = {
    "identification",
    "id",
    "rowid",
    "createdat",
    "updatedat",
    "nccreatedby",
    "ncupdatedby",
    "ncorder",
    "ncrowmeta",
    "title",
    "inscriptionid",
    "filiere",
    "niveaudetude",
    "sourcesheet",
    "etape",
    "parcours",
    "idnorm",
    "idnum"
}


def est_metadata(col):

    return (
        normaliser_nom_colonne(col)
        in META_COLS
    )


def questions_par_feuille(
    feuilles
):

    resultat = {}


    for feuille, df in feuilles.items():

        resultat[feuille] = [
            col
            for col in df.columns
            if not est_metadata(col)
        ]


    return resultat


questions_t1 = (
    questions_par_feuille(
        d3_sheets_raw
    )
)


questions_t2 = (
    questions_par_feuille(
        d4_sheets_raw
    )
)


print("T1")

for feuille, questions in questions_t1.items():

    print(
        feuille,
        ":",
        len(questions)
    )


print("\nT2")

for feuille, questions in questions_t2.items():

    print(
        feuille,
        ":",
        len(questions)
    )

T1
MI-TEST-DA : 15
MI-TEST-BI : 15
MI-TEST-DS : 15
MI-TEST-IA : 15

T2
FINAL-TEST-DA : 15
FINAL-TEST-BI : 15
FINAL-TEST-DS : 15
FINAL-TEST-IA : 15


# Cellule 13 — Complétude T1/T2

In [14]:
# ============================================================
# CELLULE 13 — Complétude par parcours
# ============================================================

def ajouter_completude(
    df,
    questions_mapping,
    prefixe
):

    resultat = df.copy()


    resultat[
        f"{prefixe}_NB_QUESTIONS"
    ] = np.nan


    resultat[
        f"{prefixe}_NB_REPONSES"
    ] = np.nan


    resultat[
        f"{prefixe}_TAUX_COMPLETUDE"
    ] = np.nan


    for feuille, questions in (
        questions_mapping.items()
    ):

        mask = (
            resultat["SOURCE_SHEET"]
            == feuille
        )


        cols = [
            col
            for col in questions
            if col in resultat.columns
        ]


        resultat.loc[
            mask,
            f"{prefixe}_NB_QUESTIONS"
        ] = len(cols)


        resultat.loc[
            mask,
            f"{prefixe}_NB_REPONSES"
        ] = (
            resultat.loc[
                mask,
                cols
            ]
            .notna()
            .sum(axis=1)
        )


        resultat.loc[
            mask,
            f"{prefixe}_TAUX_COMPLETUDE"
        ] = (
            resultat.loc[
                mask,
                f"{prefixe}_NB_REPONSES"
            ]
            / len(cols)
            * 100
        )


    return resultat


t1 = ajouter_completude(
    t1,
    questions_t1,
    "T1"
)


t2 = ajouter_completude(
    t2,
    questions_t2,
    "T2"
)


t1_clean = t1[
    t1["_ID_NORM"].notna()
].copy()


t2_clean = t2[
    t2["_ID_NORM"].notna()
].copy()


t1_sans_id = t1[
    t1["_ID_NORM"].isna()
].copy()


t2_sans_id = t2[
    t2["_ID_NORM"].isna()
].copy()


print(
    "T1 analytique :",
    len(t1_clean)
)

print(
    "T1 sans ID :",
    len(t1_sans_id)
)

print(
    "T2 analytique :",
    len(t2_clean)
)

print(
    "T2 sans ID :",
    len(t2_sans_id)
)

T1 analytique : 196
T1 sans ID : 14
T2 analytique : 206
T2 sans ID : 0


# Cellule 14 — Tables longues T1/T2

In [15]:
# ============================================================
# CELLULE 14 — Format long des réponses
# ============================================================

def construire_reponses_longues(
    feuilles,
    dictionnaire_questions,
    etape
):

    lignes = []


    for feuille, df in feuilles.items():

        parcours = (
            detecter_parcours(
                feuille
            )
        )


        id_col = (
            trouver_colonne_identifiant(
                df
            )
        )


        ids = nettoyer_identifiant(
            df[id_col]
        )


        questions = (
            dictionnaire_questions[
                feuille
            ]
        )


        for numero, question in enumerate(
            questions,
            start=1
        ):

            for index, reponse in (
                df[question].items()
            ):

                lignes.append({
                    "_ID_NORM":
                        ids.loc[index],

                    "ETAPE":
                        etape,

                    "PARCOURS":
                        parcours,

                    "SOURCE_SHEET":
                        feuille,

                    "QUESTION_INDEX":
                        numero,

                    "QUESTION":
                        question,

                    "REPONSE":
                        normaliser_valeur_objet(
                            reponse
                        ),

                    "REPONSE_PRESENTE":
                        not pd.isna(
                            reponse
                        )
                })


    return pd.DataFrame(
        lignes
    )


t1_reponses_long = (
    construire_reponses_longues(
        d3_sheets_raw,
        questions_t1,
        "T1"
    )
)


t2_reponses_long = (
    construire_reponses_longues(
        d4_sheets_raw,
        questions_t2,
        "T2"
    )
)


print(
    "T1 long :",
    t1_reponses_long.shape
)

print(
    "T2 long :",
    t2_reponses_long.shape
)


display(
    t1_reponses_long.head()
)

T1 long : (3150, 8)
T2 long : (3090, 8)


,_ID_NORM,ETAPE,PARCOURS,SOURCE_SHEET,QUESTION_INDEX,QUESTION,REPONSE,REPONSE_PRESENTE
0,<NA>,T1,DA,MI-TEST-DA,1,Qu'exprime le principe GIGO en Data Analysis ?,A) Un outil de nettoyage automatique integre d...,True
1,DIEG2026-463,T1,DA,MI-TEST-DA,1,Qu'exprime le principe GIGO en Data Analysis ?,"C) Garbage In, Garbage Out — des donnees sourc...",True
2,DIEG2026-636,T1,DA,MI-TEST-DA,1,Qu'exprime le principe GIGO en Data Analysis ?,"C) Garbage In, Garbage Out — des donnees sourc...",True
3,<NA>,T1,DA,MI-TEST-DA,1,Qu'exprime le principe GIGO en Data Analysis ?,NaN,False
4,DIEG2026-633,T1,DA,MI-TEST-DA,1,Qu'exprime le principe GIGO en Data Analysis ?,"C) Garbage In, Garbage Out — des donnees sourc...",True


# Cellule 15 — Contrôle d’unicité

In [16]:
# ============================================================
# CELLULE 15 — Unicité
# ============================================================

def controle_unicite(
    nom,
    df
):

    return {
        "Source":
            nom,

        "Lignes":
            len(df),

        "IDs_uniques":
            df["_ID_NORM"]
            .nunique(),

        "Lignes_ID_repetes":
            df["_ID_NORM"]
            .duplicated(
                keep=False
            )
            .sum()
    }


audit_unicite = pd.DataFrame([
    controle_unicite(
        "D1",
        d1_clean
    ),

    controle_unicite(
        "T0",
        t0_clean
    ),

    controle_unicite(
        "T1",
        t1_clean
    ),

    controle_unicite(
        "T2",
        t2_clean
    )
])


display(
    audit_unicite
)


if (
    audit_unicite[
        "Lignes_ID_repetes"
    ] > 0
).any():

    raise ValueError(
        "Identifiants dupliqués détectés."
    )


print(
    "✅ Une ligne par apprenant "
    "dans chaque source analytique"
)

,Source,Lignes,IDs_uniques,Lignes_ID_repetes
0,D1,472,472,0
1,T0,305,305,0
2,T1,196,196,0
3,T2,206,206,0


✅ Une ligne par apprenant dans chaque source analytique


# Cellule 16 — Cohortes finales

In [17]:
# ============================================================
# CELLULE 16 — Cohortes longitudinales
# ============================================================

IDS_D1 = set(
    d1_clean["_ID_NORM"]
)

IDS_T0 = set(
    t0_clean["_ID_NORM"]
)

IDS_T1 = set(
    t1_clean["_ID_NORM"]
)

IDS_T2 = set(
    t2_clean["_ID_NORM"]
)


COHORTE_MT0 = (
    IDS_D1
    & IDS_T0
    & IDS_T2
)


COHORTE_MT1 = (
    IDS_D1
    & IDS_T0
    & IDS_T1
    & IDS_T2
)


COHORTE_COMPARAISON = (
    set(COHORTE_MT1)
)


resume_cohortes = pd.DataFrame({

    "Cohorte": [
        "D1",
        "T0",
        "T1",
        "T2",
        "M_T0",
        "M_T1",
        "Comparaison M_T0 vs M_T1"
    ],

    "N": [
        len(IDS_D1),
        len(IDS_T0),
        len(IDS_T1),
        len(IDS_T2),
        len(COHORTE_MT0),
        len(COHORTE_MT1),
        len(COHORTE_COMPARAISON)
    ]

})


display(
    resume_cohortes
)


mt0_sans_t1 = sorted(
    COHORTE_MT0
    - COHORTE_MT1
)


print(
    "M_T0 sans T1 :",
    mt0_sans_t1
)

,Cohorte,N
0,D1,472
1,T0,305
2,T1,196
3,T2,206
4,M_T0,128
5,M_T1,125
6,Comparaison M_T0 vs M_T1,125


M_T0 sans T1 : ['DIEG2026-158', 'DIEG2026-288', 'DIEG2026-355']


# Cellule 17 — Vérification avec l’audit précédent

In [18]:
# ============================================================
# CELLULE 17 — Validation de cohérence
# ============================================================

REFERENCE = {
    "D1": 472,
    "T0": 305,
    "T1": 196,
    "T2": 206,
    "M_T0": 128,
    "M_T1": 125,
    "Comparaison": 125
}


OBSERVE = {
    "D1": len(IDS_D1),
    "T0": len(IDS_T0),
    "T1": len(IDS_T1),
    "T2": len(IDS_T2),
    "M_T0": len(COHORTE_MT0),
    "M_T1": len(COHORTE_MT1),
    "Comparaison":
        len(COHORTE_COMPARAISON)
}


controle = pd.DataFrame([

    {
        "Indicateur": nom,

        "Reference":
            REFERENCE[nom],

        "Observe":
            OBSERVE[nom],

        "Conforme":
            REFERENCE[nom]
            == OBSERVE[nom]
    }

    for nom in REFERENCE
])


display(controle)

,Indicateur,Reference,Observe,Conforme
0,D1,472,472,True
1,T0,305,305,True
2,T1,196,196,True
3,T2,206,206,True
4,M_T0,128,128,True
5,M_T1,125,125,True
6,Comparaison,125,125,True


# Cellule 18 — Préfixer les sources avant fusion

In [19]:
# ============================================================
# CELLULE 18 — Préfixage des variables
# ============================================================

def prefixer_source(
    df,
    prefixe
):

    mapping = {

        col:
            f"{prefixe}__{col}"

        for col in df.columns

        if col != "_ID_NORM"
    }


    return df.rename(
        columns=mapping
    )


d1_pref = prefixer_source(
    d1_clean,
    "D1"
)


t0_pref = prefixer_source(
    t0_clean,
    "T0"
)


t1_pref = prefixer_source(
    t1_clean,
    "T1"
)


t2_pref = prefixer_source(
    t2_clean,
    "T2"
)

# Cellule 19 — Base longitudinale complète

In [20]:
# ============================================================
# CELLULE 19 — Base longitudinale
# ============================================================

base_longitudinale = (

    d1_pref

    .merge(
        t0_pref,
        on="_ID_NORM",
        how="outer",
        validate="one_to_one"
    )

    .merge(
        t1_pref,
        on="_ID_NORM",
        how="outer",
        validate="one_to_one"
    )

    .merge(
        t2_pref,
        on="_ID_NORM",
        how="outer",
        validate="one_to_one"
    )

)


base_longitudinale[
    "PRESENT_D1"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(IDS_D1)
)


base_longitudinale[
    "PRESENT_T0"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(IDS_T0)
)


base_longitudinale[
    "PRESENT_T1"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(IDS_T1)
)


base_longitudinale[
    "PRESENT_T2"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(IDS_T2)
)


base_longitudinale[
    "COHORTE_MT0"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(COHORTE_MT0)
)


base_longitudinale[
    "COHORTE_MT1"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(COHORTE_MT1)
)


base_longitudinale[
    "COHORTE_COMPARAISON"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .isin(
        COHORTE_COMPARAISON
    )
)


base_longitudinale[
    "ID_NUM"
] = (
    base_longitudinale[
        "_ID_NORM"
    ]
    .map(
        extraire_numero_id
    )
)


base_longitudinale = (
    base_longitudinale
    .sort_values(
        "ID_NUM"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Base longitudinale :",
    base_longitudinale.shape
)


print(
    "Apprenants uniques :",
    base_longitudinale[
        "_ID_NORM"
    ]
    .nunique()
)

Base longitudinale : (532, 208)
Apprenants uniques : 532


# Cellule 20 — Construire les bases M_T0 et M_T1

Important : ce sont encore des tables d’intégration. Elles contiennent T2 pour construire ultérieurement la cible.

In [21]:
# ============================================================
# CELLULE 20 — Bases par cadre prédictif
# ============================================================

cols_mt0 = (

    [
        "_ID_NORM",
        "ID_NUM"
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("D1__")
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("T0__")
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("T2__")
    ]

)


base_mt0_sources = (

    base_longitudinale.loc[

        base_longitudinale[
            "COHORTE_MT0"
        ],

        cols_mt0

    ]

    .copy()

)


cols_mt1 = (

    [
        "_ID_NORM",
        "ID_NUM"
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("D1__")
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("T0__")
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("T1__")
    ]

    + [
        c
        for c in base_longitudinale.columns
        if c.startswith("T2__")
    ]

)


base_mt1_sources = (

    base_longitudinale.loc[

        base_longitudinale[
            "COHORTE_MT1"
        ],

        cols_mt1

    ]

    .copy()

)


print(
    "M_T0 :",
    base_mt0_sources.shape
)


print(
    "M_T1 :",
    base_mt1_sources.shape
)

M_T0 : (128, 125)
M_T1 : (125, 201)


# Cellule 21 — Index longitudinal synthétique

In [22]:
# ============================================================
# CELLULE 21 — Index longitudinal
# ============================================================

index_longitudinal = (

    base_longitudinale[
        [
            "_ID_NORM",
            "ID_NUM",

            "PRESENT_D1",
            "PRESENT_T0",
            "PRESENT_T1",
            "PRESENT_T2",

            "COHORTE_MT0",
            "COHORTE_MT1",
            "COHORTE_COMPARAISON"
        ]
    ]

    .copy()

)


def statut_longitudinal(row):

    if row["COHORTE_MT1"]:

        return "M_T1 / comparaison"


    if row["COHORTE_MT0"]:

        return "M_T0 sans T1"


    if (
        row["PRESENT_T1"]
        and row["PRESENT_T2"]
        and not row["PRESENT_D1"]
    ):

        return "T1-T2 sans D1"


    if (
        row["PRESENT_T2"]
        and not row["PRESENT_D1"]
    ):

        return "T2 sans D1"


    return (
        "Hors cohortes prédictives"
    )


index_longitudinal[
    "STATUT_LONGITUDINAL"
] = (
    index_longitudinal
    .apply(
        statut_longitudinal,
        axis=1
    )
)


display(

    index_longitudinal[
        "STATUT_LONGITUDINAL"
    ]

    .value_counts()

    .rename("N")

    .to_frame()

)

,N
STATUT_LONGITUDINAL,
Hors cohortes prédictives,344
M_T1 / comparaison,125
T1-T2 sans D1,55
T2 sans D1,5
M_T0 sans T1,3


# Cellule 22 — Rapport qualité

In [23]:
# ============================================================
# CELLULE 22 — Rapport de qualité
# ============================================================

rapport_qualite = pd.DataFrame([

    {
        "Source": "D1",
        "Lignes_brutes":
            len(d1_raw),
        "Lignes_analytiques":
            len(d1_clean),
        "Sans_ID":
            d1["_ID_NORM"]
            .isna()
            .sum(),
        "IDs_uniques":
            d1_clean[
                "_ID_NORM"
            ].nunique()
    },

    {
        "Source": "T0",
        "Lignes_brutes":
            len(d2_raw),
        "Lignes_analytiques":
            len(t0_clean),
        "Sans_ID":
            len(t0_sans_id),
        "IDs_uniques":
            t0_clean[
                "_ID_NORM"
            ].nunique()
    },

    {
        "Source": "T1",
        "Lignes_brutes":
            len(t1_raw),
        "Lignes_analytiques":
            len(t1_clean),
        "Sans_ID":
            len(t1_sans_id),
        "IDs_uniques":
            t1_clean[
                "_ID_NORM"
            ].nunique()
    },

    {
        "Source": "T2",
        "Lignes_brutes":
            len(t2_raw),
        "Lignes_analytiques":
            len(t2_clean),
        "Sans_ID":
            len(t2_sans_id),
        "IDs_uniques":
            t2_clean[
                "_ID_NORM"
            ].nunique()
    }

])


display(
    rapport_qualite
)

,Source,Lignes_brutes,Lignes_analytiques,Sans_ID,IDs_uniques
0,D1,472,472,0,472
1,T0,315,305,10,305
2,T1,210,196,14,196
3,T2,206,206,0,206


# Cellule 23 — Export de tous les résultats

In [24]:
# ============================================================
# CELLULE 23 — Exports
# ============================================================

exports = {

    "D1_profils_nettoyes.csv":
        d1_clean,

    "T0_nettoye.csv":
        t0_clean,

    "T1_nettoye.csv":
        t1_clean,

    "T2_nettoye.csv":
        t2_clean,

    "T0_sans_identifiant.csv":
        t0_sans_id,

    "T1_sans_identifiant.csv":
        t1_sans_id,

    "T2_sans_identifiant.csv":
        t2_sans_id,

    "dictionnaire_questions_T0.csv":
        dictionnaire_t0,

    "T1_reponses_long.csv":
        t1_reponses_long,

    "T2_reponses_long.csv":
        t2_reponses_long,

    "base_longitudinale_complete.csv":
        base_longitudinale,

    "base_MT0_sources.csv":
        base_mt0_sources,

    "base_MT1_sources.csv":
        base_mt1_sources,

    "index_longitudinal.csv":
        index_longitudinal,

    "audit_unicite_apres_nettoyage.csv":
        audit_unicite,

    "resume_cohortes_apres_nettoyage.csv":
        resume_cohortes,

    "rapport_qualite_apres_nettoyage.csv":
        rapport_qualite
}


for nom_fichier, dataframe in exports.items():

    dataframe.to_csv(

        OUTPUT_DIR
        / nom_fichier,

        index=False,

        encoding="utf-8-sig"

    )


print(
    f"✅ {len(exports)} fichiers CSV sauvegardés"
)

print(
    OUTPUT_DIR
)

✅ 17 fichiers CSV sauvegardés
/home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/01_nettoyage


# Cellule 24 — Synthèse JSON

In [25]:
# ============================================================
# CELLULE 24 — Synthèse reproductible
# ============================================================

import hashlib
import platform
import sklearn


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for bloc in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloc)

    return h.hexdigest()


HASH_SOURCES = {
    "D1":
        sha256_file(D1_FILE),

    "T0":
        sha256_file(D2_FILE),

    "T1":
        sha256_file(D3_FILE),

    "T2":
        sha256_file(D4_FILE)
}


synthese = {

    "sources": {

        "D1":
            D1_FILE.name,

        "T0":
            D2_FILE.name,

        "T1":
            D3_FILE.name,

        "T2":
            D4_FILE.name
    },


    "hash_sha256":
        HASH_SOURCES,


    "versions": {

        "python":
            platform.python_version(),

        "pandas":
            pd.__version__,

        "numpy":
            np.__version__,

        "sklearn":
            sklearn.__version__
    },


    "effectifs": {

        "D1":
            len(IDS_D1),

        "T0":
            len(IDS_T0),

        "T1":
            len(IDS_T1),

        "T2":
            len(IDS_T2),

        "M_T0":
            len(COHORTE_MT0),

        "M_T1":
            len(COHORTE_MT1),

        "comparaison":
            len(
                COHORTE_COMPARAISON
            )
    },


    "sans_identifiant": {

        "T0":
            len(t0_sans_id),

        "T1":
            len(t1_sans_id),

        "T2":
            len(t2_sans_id)
    },


    "MT0_sans_T1":
        mt0_sans_t1,


    "regles": {

        "imputation":
            False,

        "scoring":
            False,

        "modelisation":
            False,

        "T2_predicteur":
            False,

        "nan_structurels_imputes":
            False,

        "comparaison_MT0_MT1_meme_cohorte":
            True
    }

}


with open(

    OUTPUT_DIR
    / "synthese_01_integration_nettoyage.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        synthese,

        f,

        ensure_ascii=False,

        indent=4

    )


print(
    json.dumps(
        synthese,
        ensure_ascii=False,
        indent=4
    )
)

{
    "sources": {
        "D1": "inscriptions.json",
        "T0": "tests.json",
        "T1": "mi-test.xlsx",
        "T2": "RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx"
    },
    "hash_sha256": {
        "D1": "c683734c9e59b097f51bc9f2590c61f9b546d0b7c98acc711bec612c18df8b2a",
        "T0": "1042ac564a679e7814e59dc92ad7774087e1075220d93b4e9c6e6821fe4c9617",
        "T1": "0db353948a83c195baeb898ef1814b03896fa6d10ca62f15de8688805da35b9a",
        "T2": "b0e59e2ec6d94bad70b9321c3711971bbc6448ae73c53ec7a0f37b3cf066fa58"
    },
    "versions": {
        "python": "3.12.3",
        "pandas": "2.1.4",
        "numpy": "1.26.4",
        "sklearn": "1.3.2"
    },
    "effectifs": {
        "D1": 472,
        "T0": 305,
        "T1": 196,
        "T2": 206,
        "M_T0": 128,
        "M_T1": 125,
        "comparaison": 125
    },
    "sans_identifiant": {
        "T0": 10,
        "T1": 14,
        "T2": 0
    },
    "MT0_sans_T1": [
        "DIEG2026-158",
        "DIEG2026-288",
        "DI

# Cellule 25 — Vérification finale

In [26]:
# ============================================================
# CELLULE 25 — Vérification finale
# ============================================================

print("=" * 80)

print(
    "SYNTHÈSE — NOTEBOOK 01"
)

print("=" * 80)


print(
    "D1 :",
    len(IDS_D1)
)

print(
    "T0 :",
    len(IDS_T0)
)

print(
    "T1 :",
    len(IDS_T1)
)

print(
    "T2 :",
    len(IDS_T2)
)


print()


print(
    "M_T0 :",
    len(COHORTE_MT0)
)

print(
    "M_T1 :",
    len(COHORTE_MT1)
)

print(
    "Comparaison :",
    len(COHORTE_COMPARAISON)
)


print()


print(
    "M_T0 sans T1 :",
    mt0_sans_t1
)


print()


print(
    "Dossier outputs :",
    OUTPUT_DIR
)


# Contrôles d'intégrité des effectifs
assert len(IDS_D1) == 472
assert len(IDS_T0) == 305
assert len(IDS_T1) == 196
assert len(IDS_T2) == 206

assert len(COHORTE_MT0) == 128
assert len(COHORTE_MT1) == 125
assert len(COHORTE_COMPARAISON) == 125


print("\n✅ NOTEBOOK 01 TERMINÉ")

SYNTHÈSE — NOTEBOOK 01
D1 : 472
T0 : 305
T1 : 196
T2 : 206

M_T0 : 128
M_T1 : 125
Comparaison : 125

M_T0 sans T1 : ['DIEG2026-158', 'DIEG2026-288', 'DIEG2026-355']

Dossier outputs : /home/andrycomputer/Documents/Model-eval-pers-elearning/outputs/01_nettoyage

✅ NOTEBOOK 01 TERMINÉ
